<a href="https://colab.research.google.com/github/Rinosa123/Bilingual-Enterprise-RAG-Copilot/blob/main/notebooks/03_multilingual_reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic–English Multilingual Reranking

This notebook adds a multilingual cross-encoder reranker after candidate retrieval.

Pipeline:

1. Retrieve candidate chunks using multilingual dense retrieval.
2. Score every query–chunk pair using BGE Reranker v2 M3.
3. Reorder candidates using the cross-encoder scores.
4. Compare dense retrieval against reranked retrieval.

In [ ]:
%cd /content

!if [ -d "Bilingual-Enterprise-RAG-Copilot/.git" ]; then \
    git -C Bilingual-Enterprise-RAG-Copilot pull --ff-only origin main; \
else \
    git clone https://github.com/Rinosa123/Bilingual-Enterprise-RAG-Copilot.git; \
fi

%cd /content/Bilingual-Enterprise-RAG-Copilot

!pip -q install sentence-transformers FlagEmbedding

## Load Documents, Dense Retriever and Cross-Encoder Reranker

In [ ]:
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch
from FlagEmbedding import FlagReranker
from sentence_transformers import SentenceTransformer

from src.ingestion.chunker import chunk_documents
from src.ingestion.text_loader import load_text_documents


PROJECT_ROOT = Path.cwd()
DOCUMENT_DIRECTORY = PROJECT_ROOT / "data" / "sample_docs"

DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"


# Load the demonstration documents.
documents = load_text_documents(
    DOCUMENT_DIRECTORY
)

chunks = chunk_documents(documents)


# Load the multilingual dense retrieval model.
dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    device="cuda",
)


# Load the multilingual cross-encoder reranker.
reranker = FlagReranker(
    RERANKER_MODEL_NAME,
    devices=["cuda:0"],
    use_fp16=True,
)


print("FlagEmbedding:", version("FlagEmbedding"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Dense model:", DENSE_MODEL_NAME)
print("Dense-model device:", dense_model.device)
print("Reranker:", RERANKER_MODEL_NAME)
print("Documents:", len(documents))
print("Chunks:", len(chunks))

## Dense Candidate Retrieval

The dense retriever selects five candidate chunks for each question. The cross-encoder will rerank only these candidates.

In [3]:
# Create dense embeddings for all document chunks.
passage_texts = [
    f"passage: {chunk.section}\n{chunk.text}"
    for chunk in chunks
]

passage_embeddings = dense_model.encode(
    passage_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)


EVALUATION_QUERIES = (
    (
        "English question -> English document",
        "How many annual leave days do full-time employees receive?",
        "HR-EN-001-CH-003",
    ),
    (
        "Arabic question -> Arabic document",
        "ما الحد الأقصى لتكلفة الفندق؟",
        "HR-AR-001-CH-003",
    ),
    (
        "Arabic question -> English document",
        "كم عدد أيام الإجازة السنوية للموظف؟",
        "HR-EN-001-CH-003",
    ),
    (
        "English question -> Arabic document",
        "When must an expense claim be submitted?",
        "HR-AR-001-CH-002",
    ),
)


query_texts = [
    f"query: {question}"
    for _, question, _ in EVALUATION_QUERIES
]

query_embeddings = dense_model.encode(
    query_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
)


similarity_matrix = (
    query_embeddings
    @ passage_embeddings.T
)

CANDIDATE_COUNT = 5
candidate_sets = []


for query_index, (
    label,
    question,
    expected_chunk_id,
) in enumerate(EVALUATION_QUERIES):

    scores = similarity_matrix[query_index]
    ranked_indices = np.argsort(-scores)

    dense_ranking = [
        chunks[int(index)].chunk_id
        for index in ranked_indices
    ]

    candidate_chunks = [
        chunks[int(index)]
        for index in ranked_indices[:CANDIDATE_COUNT]
    ]

    expected_dense_rank = (
        dense_ranking.index(expected_chunk_id) + 1
    )

    expected_in_candidates = any(
        chunk.chunk_id == expected_chunk_id
        for chunk in candidate_chunks
    )

    candidate_sets.append(
        {
            "label": label,
            "question": question,
            "expected": expected_chunk_id,
            "dense_ranking": dense_ranking,
            "candidate_chunks": candidate_chunks,
        }
    )

    print("=" * 80)
    print("Test:", label)
    print("Expected chunk:", expected_chunk_id)
    print("Expected dense rank:", expected_dense_rank)
    print(
        "Expected in Top 5:",
        expected_in_candidates,
    )
    print(
        "Dense candidates:",
        [
            chunk.chunk_id
            for chunk in candidate_chunks
        ],
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test: English question -> English document
Expected chunk: HR-EN-001-CH-003
Expected dense rank: 1
Expected in Top 5: True
Dense candidates: ['HR-EN-001-CH-003', 'HR-EN-001-CH-002', 'HR-EN-001-CH-004', 'HR-AR-001-CH-005', 'HR-AR-001-CH-004']
Test: Arabic question -> Arabic document
Expected chunk: HR-AR-001-CH-003
Expected dense rank: 1
Expected in Top 5: True
Dense candidates: ['HR-AR-001-CH-003', 'HR-AR-001-CH-002', 'HR-AR-001-CH-005', 'HR-AR-001-CH-004', 'HR-AR-001-CH-001']
Test: Arabic question -> English document
Expected chunk: HR-EN-001-CH-003
Expected dense rank: 4
Expected in Top 5: True
Dense candidates: ['HR-AR-001-CH-004', 'HR-AR-001-CH-003', 'HR-AR-001-CH-005', 'HR-EN-001-CH-003', 'HR-AR-001-CH-002']
Test: English question -> Arabic document
Expected chunk: HR-AR-001-CH-002
Expected dense rank: 1
Expected in Top 5: True
Dense candidates: ['HR-AR-001-CH-002', 'HR-EN-001-CH-001', 'HR-EN-001-CH-003', 'HR-AR-001-CH-004', 'HR-EN-001-CH-005']


## Cross-Encoder Reranking

BGE Reranker scores each question and candidate passage together. Higher scores indicate greater relevance.

In [9]:
reranking_results = []


for candidate_set in candidate_sets:
    question = candidate_set["question"]
    expected_chunk_id = candidate_set["expected"]
    candidate_chunks = candidate_set["candidate_chunks"]

    # Create question–passage pairs for the cross-encoder.
    question_passage_pairs = [
        [
            question,
            f"{chunk.section}\n{chunk.text}",
        ]
        for chunk in candidate_chunks
    ]

    # Convert raw model scores to values between 0 and 1.
    reranker_scores = compute_reranker_scores(
        question_passage_pairs
    )

    reranker_scores = np.asarray(
        reranker_scores,
        dtype=float,
    )

    # Sort candidates from highest to lowest relevance.
    reranked_indices = np.argsort(
        -reranker_scores
    )

    reranked_chunks = [
        candidate_chunks[int(index)]
        for index in reranked_indices
    ]

    reranked_scores = [
        float(reranker_scores[int(index)])
        for index in reranked_indices
    ]

    reranked_ranking = [
        chunk.chunk_id
        for chunk in reranked_chunks
    ]

    expected_reranked_rank = (
        reranked_ranking.index(expected_chunk_id) + 1
    )

    reranking_results.append(
        {
            "label": candidate_set["label"],
            "question": question,
            "expected": expected_chunk_id,
            "dense_ranking": candidate_set[
                "dense_ranking"
            ],
            "reranked_ranking": reranked_ranking,
            "reranked_scores": reranked_scores,
        }
    )

    print("=" * 80)
    print("Test:", candidate_set["label"])
    print("Question:", question)
    print("Expected:", expected_chunk_id)
    print(
        "Expected reranked rank:",
        expected_reranked_rank,
    )
    print("Reranked candidates:")

    for rank, (
        chunk,
        score,
    ) in enumerate(
        zip(
            reranked_chunks,
            reranked_scores,
        ),
        start=1,
    ):
        print(
            f"  {rank}. {chunk.chunk_id} | "
            f"score={score:.4f} | "
            f"section={chunk.section}"
        )

Test: English question -> English document
Question: How many annual leave days do full-time employees receive?
Expected: HR-EN-001-CH-003
Expected reranked rank: 1
Reranked candidates:
  1. HR-EN-001-CH-003 | score=0.9981 | section=2. Annual Leave
  2. HR-AR-001-CH-005 | score=0.0042 | section=4. التدريب والتطوير المهني
  3. HR-AR-001-CH-004 | score=0.0015 | section=3. الإجازة المرضية
  4. HR-EN-001-CH-004 | score=0.0011 | section=3. Remote Work
  5. HR-EN-001-CH-002 | score=0.0004 | section=1. Working Hours
Test: Arabic question -> Arabic document
Question: ما الحد الأقصى لتكلفة الفندق؟
Expected: HR-AR-001-CH-003
Expected reranked rank: 1
Reranked candidates:
  1. HR-AR-001-CH-003 | score=0.8430 | section=2. السفر في مهام العمل
  2. HR-AR-001-CH-002 | score=0.0005 | section=1. مطالبات المصروفات
  3. HR-AR-001-CH-005 | score=0.0000 | section=4. التدريب والتطوير المهني
  4. HR-AR-001-CH-004 | score=0.0000 | section=3. الإجازة المرضية
  5. HR-AR-001-CH-001 | score=0.0000 | section=معلوم

In [6]:
def compute_reranker_scores(
    question_passage_pairs: list[list[str]],
) -> np.ndarray:
    """Score query–passage pairs using the loaded BGE model."""

    questions = [
        pair[0]
        for pair in question_passage_pairs
    ]

    passages = [
        pair[1]
        for pair in question_passage_pairs
    ]

    encoded_pairs = reranker.tokenizer(
        questions,
        passages,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    model_device = next(
        reranker.model.parameters()
    ).device

    encoded_pairs = {
        name: tensor.to(model_device)
        for name, tensor in encoded_pairs.items()
    }

    reranker.model.eval()

    with torch.inference_mode():
        logits = reranker.model(
            **encoded_pairs,
            return_dict=True,
        ).logits.view(-1).float()

    normalized_scores = torch.sigmoid(logits)

    return (
        normalized_scores
        .cpu()
        .numpy()
    )


print("Compatibility helper created successfully.")

Compatibility helper created successfully.


## Dense Retrieval vs Cross-Encoder Reranking

In [10]:
import pandas as pd


def find_rank(
    ranking: list[str],
    expected_chunk_id: str,
) -> int | None:
    """Return the one-based rank of the expected chunk."""

    try:
        return ranking.index(expected_chunk_id) + 1
    except ValueError:
        return None


rank_rows = []


for result in reranking_results:
    dense_rank = find_rank(
        result["dense_ranking"],
        result["expected"],
    )

    reranked_rank = find_rank(
        result["reranked_ranking"],
        result["expected"],
    )

    rank_rows.append(
        {
            "Test": result["label"],
            "Dense Rank": dense_rank,
            "Reranked Rank": reranked_rank,
            "Rank Improvement": (
                dense_rank - reranked_rank
                if dense_rank is not None
                and reranked_rank is not None
                else None
            ),
        }
    )


rank_table = pd.DataFrame(rank_rows)

display(rank_table)


def calculate_metrics(
    ranking_key: str,
    display_name: str,
) -> dict[str, float | str]:
    """Calculate Top-1, Hit@3 and MRR."""

    ranks = [
        find_rank(
            result[ranking_key],
            result["expected"],
        )
        for result in reranking_results
    ]

    query_count = len(ranks)

    top_1_accuracy = sum(
        rank == 1
        for rank in ranks
    ) / query_count

    hit_at_3 = sum(
        rank is not None and rank <= 3
        for rank in ranks
    ) / query_count

    mean_reciprocal_rank = sum(
        1 / rank
        if rank is not None
        else 0
        for rank in ranks
    ) / query_count

    return {
        "Retriever": display_name,
        "Top-1 Accuracy": top_1_accuracy,
        "Hit@3": hit_at_3,
        "MRR": mean_reciprocal_rank,
    }


metrics_table = pd.DataFrame(
    [
        calculate_metrics(
            "dense_ranking",
            "Dense E5",
        ),
        calculate_metrics(
            "reranked_ranking",
            "Dense E5 + BGE Reranker",
        ),
    ]
)


display(
    metrics_table.style.format(
        {
            "Top-1 Accuracy": "{:.0%}",
            "Hit@3": "{:.0%}",
            "MRR": "{:.4f}",
        }
    )
)


print("\nPlain-text summary:")

for _, row in metrics_table.iterrows():
    print(
        f"{row['Retriever']}: "
        f"Top-1={row['Top-1 Accuracy']:.0%}, "
        f"Hit@3={row['Hit@3']:.0%}, "
        f"MRR={row['MRR']:.4f}"
    )

,Test,Dense Rank,Reranked Rank,Rank Improvement
0,English question -> English document,1,1,0
1,Arabic question -> Arabic document,1,1,0
2,Arabic question -> English document,4,1,3
3,English question -> Arabic document,1,1,0


,Retriever,Top-1 Accuracy,Hit@3,MRR
0,Dense E5,75%,75%,0.8125
1,Dense E5 + BGE Reranker,100%,100%,1.0000



Plain-text summary:
Dense E5: Top-1=75%, Hit@3=75%, MRR=0.8125
Dense E5 + BGE Reranker: Top-1=100%, Hit@3=100%, MRR=1.0000


## Findings and Engineering Decision

- Dense retrieval achieved 75% Top-1 accuracy, 75% Hit@3 and an MRR of 0.8125.
- Dense Top-5 candidate retrieval included the expected chunk for all four questions.
- BGE Reranker v2 M3 improved the Arabic-to-English annual-leave result from rank 4 to rank 1.
- After reranking, all four expected chunks were ranked first, producing 100% Top-1 accuracy, 100% Hit@3 and an MRR of 1.0000.

### Production Retrieval Decision

The production retrieval pipeline will use:

1. Multilingual E5 to retrieve candidate chunks.
2. Optional BM25 retrieval for exact keywords, identifiers and policy codes.
3. BGE Reranker v2 M3 to reorder the candidate set.
4. The highest-ranked chunks as grounded context for answer generation.

### Compatibility Note

FlagEmbedding 1.4.0 currently calls a tokenizer method removed in Transformers v5. This notebook therefore uses the loaded BGE tokenizer and model directly for query–passage scoring. The reranker model itself is unchanged.

### Limitations

This initial benchmark contains two synthetic bilingual policy documents and four questions. The 100% result demonstrates that the pipeline works on this controlled example; it should not be interpreted as production-level accuracy. A larger bilingual evaluation dataset is required.